# Day 6 lab: Data skew

**Goal:** Spot skew in the Spark UI, explain why it hurts, and choose between AQE, broadcasting, salting, hot-key isolation and repartitioning.

**Time:** about 2 h 30 min. Checklist: 4.1–4.11, 6.2.

Put [`lab_utils.py`](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/lab_utils.py) next to this notebook, run the setup cell, then work through the tasks in order. Write your code in the empty cell under each task and your observations in the notes cell. Keep the Spark UI open (the setup cell prints its address).

In [ ]:
from lab_utils import *

spark = get_spark()

## 📘 Code along

Type the code yourself instead of pasting it, run every cell, and keep the Spark UI open. Swap in `spark = get_spark()` from the starter kit wherever an example builds its own session.

### Task 1. Partitioning tools in the API reference — 30 min

Type the Examples sections of the official API pages for `DataFrame.repartition`, `repartitionByRange`, `coalesce` and `hint`, and of `functions.broadcast`, then the SQL partitioning hints (`REPARTITION`, `REBALANCE`, `COALESCE`) from Performance Tuning. After each one, print `df.rdd.getNumPartitions()` and `partition_sizes(df)`, and look for the Exchange (or its absence) in `explain()`.

[Partitioning tools in the API reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.repartition.html) · [repartitionByRange](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.repartitionByRange.html) · [coalesce](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.coalesce.html) · [hint](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.hint.html) · [broadcast](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.broadcast.html) · [SQL partitioning hints](https://spark.apache.org/docs/latest/sql-performance-tuning.html#coalesce-hints-for-sql-queries)

In [ ]:
# Task 1: your code here


*Task 1 notes: what you predicted, what the plan and the Spark UI showed.*

## ✍️ Exercises

Write these from scratch in the cells below, using the patterns you just typed. Each one ends with a Check you can verify yourself; open the reference solution only after you have a working answer.

### Task 2. Exercise 1: salt a skewed join — 45 min

Write `make_orders(spark, 4_000_000, hot_share=0.5)` to `data/orders_skewed` (half the rows have customer_id 0). With AQE off and broadcasting off (`spark.sql.autoBroadcastJoinThreshold` = -1), join it with customers and find the straggler in the Stages tab: Max vs Median duration and shuffle read. Then (a) salt every key: `salt = F.floor(F.rand(1) * 16)` on the orders side and `F.explode(F.sequence(F.lit(0), F.lit(15)))` on customers, joined on ["customer_id", "salt"]; (b) salt only the hot key, keeping salt 0 for every other key and replicating only customer 0. 

**Check:** all three joins return the same number of rows; after hashing into 64 partitions the max ÷ median row ratio drops from about 65 to about 9, and the joins get faster; (a) grows customers to 160,000 rows, (b) to 10,015. Proves 4.2, 4.3, 4.5, 4.11.

[Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/day06.py)

In [ ]:
# Task 2: your code here


*Task 2 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 3. Exercise 2: a two-stage distinct count — 30 min

On the skewed data, `groupBy("customer_id").agg(F.countDistinct("product_id"))` sends the whole hot key to one task (check the task metrics). Rewrite it in two stages: add `salt = F.pmod(F.hash("product_id"), F.lit(16))` so each product lands in exactly one salt, count distinct per (customer_id, salt), then sum per customer_id. AQE skew handling does not help aggregations, so this is on you. 

**Check:** the result equals the exact one for every customer (the hot key has 1,000 distinct products). Proves 4.4, 4.6.

[Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/day06.py)

In [ ]:
# Task 3: your code here


*Task 3 notes: what you predicted, what the plan and the Spark UI showed.*

## 🔬 Prove it

Short, focused checks of today's checklist questions in the plan and the Spark UI.

### Task 4. AQE skew join — 25 min

AQE on, broadcasting still off. For local data, lower the thresholds: `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` = `4MB` and `spark.sql.adaptive.advisoryPartitionSizeInBytes` = `4MB`. Build the skewed join with an aggregate on top (`groupBy("country").agg(F.count("note"))`), `collect()` it and `explain()`: look for `SortMergeJoin(skew=true)`, an `AQEShuffleRead` marked `skewed`, and the split tasks in the UI. Then aggregate only `customer_id`, so that just the key is shuffled: the skew goes undetected, because AQE measures bytes, not rows. Proves 4.6.

In [ ]:
# Task 4: your code here


*Task 4 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 5. repartition vs coalesce — 20 min

Print `partition_sizes()` for `s.repartition(8)`, `s.repartition(8, "customer_id")`, `s.repartition(8).coalesce(2)` and `s.repartition(8, "customer_id").coalesce(2)`. Expect even sizes; one partition of about 2.2 million rows; two even halves; and two uneven partitions (about 3 million and 1 million): coalesce only glues partitions together, so it cannot undo skew. Then write `s` as one gzip CSV (`coalesce(1)`, `option("compression", "gzip")`) and read it back: one read task however many cores you have, which is read-side skew. Proves 4.8, 4.9, 4.10, 6.2.

In [ ]:
# Task 5: your code here


*Task 5 notes: what you predicted, what the plan and the Spark UI showed.*